# 07 — Controlled simulation validation

Known innovation regimes test whether each residual model succeeds for the reason claimed. The heteroskedastic design is particularly important for the volatility-standardized benchmark. DI-VAR is optional here because notebook 09 provides the primary repeated-seed comparison.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
CACHE = ROOT / 'results/notebook_cache'
CACHE.mkdir(parents=True, exist_ok=True)
from innovcal.data.simulation import generate_multiple_var_datasets, make_stable_var_matrix
from innovcal.experiments.financial import run_financial_experiment

In [2]:
K = 4
A = make_stable_var_matrix(K, scale=0.4, seed=10)
Sigma = 0.2 * np.ones((K, K)) + 0.8 * np.eye(K)
datasets = generate_multiple_var_datasets(
    ['gaussian', 'student_t', 'mixture', 'heteroskedastic'],
    n_obs=500,
    burn_in=100,
    A=A,
    Sigma=Sigma,
    base_seed=20,
)
RUN_DIFFUSION = False
methods = ['gaussian', 'student_t', 'bootstrap', 'block_bootstrap', 'volatility_bootstrap']
if RUN_DIFFUSION:
    methods.append('diffusion')

/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/data/simulation.py:47: RuntimeWarning: divide by zero encountered in matmul
  return rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/data/simulation.py:47: RuntimeWarning: overflow encountered in matmul
  return rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/data/simulation.py:47: RuntimeWarning: invalid value encountered in matmul
  return rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/data/simulation.py:66: RuntimeWarning: divide by zero encountered in matmul
  z = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/data/simulation.py:66: RuntimeWarning: overflow encountered in matmul
  z = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/data/simulation.py:66: RuntimeWarning: invalid value encountered in

In [3]:
tables = []
for name, dataset in datasets.items():
    result = run_financial_experiment(
        dataset['y'],
        methods=tuple(methods),
        lags=1,
        n_paths=200,
        seed=123,
        diffusion_options={
            'timesteps': 50,
            'epochs': 100,
            'hidden_dim': 64,
            'validation_fraction': 0.2,
            'early_stopping_patience': 20,
            'verbose': False,
        },
    )
    table = result.evaluation.copy()
    table['dgp'] = name
    tables.append(table)

simulation_results = pd.concat(tables, ignore_index=True)
display(simulation_results.sort_values(['dgp', 'energy_score']))

/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: divide by zero encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: overflow encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: invalid value encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: divide by zero encountered in matmul
  shocks = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: overflow encountered in matmul
  shocks = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: invalid value encountered in matmul
  shocks = rng.

,dgp,forecast_model,innovation_model,avg_coverage,avg_width,energy_score,crps,interval_score,ece,pit_deviation,...,width_1,coverage_2,width_2,coverage_3,width_3,coverage_4,width_4,nominal_coverage,coverage_error,abs_coverage_error
1,gaussian,VAR,student_t,0.8900,3.310989,1.383937,0.589557,4.563696,0.030833,0.0180,...,2.938031,0.86,2.947686,0.93,3.576028,0.91,3.782211,0.9,-0.0100,0.0100
2,gaussian,VAR,bootstrap,0.8800,3.462249,1.392267,0.591263,4.619466,0.019167,0.0150,...,3.004081,0.85,3.007839,0.91,3.844497,0.92,3.992580,0.9,-0.0200,0.0200
0,gaussian,VAR,gaussian,0.8950,3.454778,1.398041,0.596534,4.600461,0.008333,0.0175,...,3.049171,0.86,3.049414,0.91,3.765237,0.96,3.955289,0.9,-0.0050,0.0050
3,gaussian,VAR,block_bootstrap,0.8850,3.369893,1.399063,0.594286,4.533786,0.006667,0.0150,...,3.238253,0.82,2.808064,0.91,3.632806,0.91,3.800450,0.9,-0.0150,0.0150
4,gaussian,VAR,volatility_bootstrap,0.9450,4.533221,1.420645,0.605665,5.009826,0.065833,0.0210,...,3.858732,0.97,4.881139,0.90,3.928756,0.99,5.464256,0.9,0.0450,0.0450
18,heteroskedastic,VAR,block_bootstrap,0.9025,6.325420,1.925421,0.823623,8.681762,0.006667,0.0090,...,6.450587,0.90,6.183084,0.91,5.846746,0.91,6.821262,0.9,0.0025,0.0025
17,heteroskedastic,VAR,bootstrap,0.8950,6.089915,1.942438,0.834043,8.622385,0.033333,0.0200,...,6.572911,0.87,5.708287,0.90,5.772695,0.89,6.305768,0.9,-0.0050,0.0050
16,heteroskedastic,VAR,student_t,0.8875,5.582867,2.003434,0.856605,8.445885,0.081667,0.0515,...,5.567428,0.90,5.876020,0.91,5.089139,0.89,5.798880,0.9,-0.0125,0.0125
15,heteroskedastic,VAR,gaussian,0.8950,5.855341,2.053411,0.875292,8.280790,0.095000,0.0570,...,5.843226,0.90,6.181138,0.91,5.324197,0.89,6.072805,0.9,-0.0050,0.0050
19,heteroskedastic,VAR,volatility_bootstrap,0.9275,7.402131,2.066689,0.880186,8.649174,0.106667,0.0565,...,8.699840,0.89,6.825056,0.94,7.074876,0.94,7.008753,0.9,0.0275,0.0275


In [4]:
simulation_results.to_csv(CACHE / 'simulation_results.csv', index=False)